# AUT – Alternative Uses Task

Generate a creative-uses prompt for a common object.

1. Call `instruct.instruct(cue=None, seed=None)`. If `cue` is omitted a random object is sampled from the standard list.
2. The returned `response_format` already contains the original `cue` so later evaluation has full context.
3. Collect a list of short creative-use phrases. 
4. Evaluation using SemDis from Beaty and Johnson, 2021

In [1]:
import instruct, evaluate
from glove_word_embeddings import mod, pre
import numpy as np

# Instruction

In [2]:
stim = instruct.instruct(seed=0)
for k, v in stim.items():
    print(k,v,'\n')

stim_brick = instruct.instruct(cue="brick")
print("cue =",stim_brick["cue"])

test aut 

cue comb 

n_words None 

instructions What are some creative uses for this object: comb?

The goal is to come up with creative uses, which are ideas that may strike as clever, unusual, interesting, uncommon, humorous, innovative, or different.

Rules:
1. List as many creative uses as you can.
2. Answer in short phrases, one use per line.
3. Return only the list of uses, nothing else.
4. Do not return any thought process or explanations other than the list of uses.

Notes:
Return the uses as a plain list (one per line). Do not return anything else. 

response_format {'cue': 'comb', 'responses': ['use 1', 'use 2', '...']} 

cue = brick


# Evaluation
Expected response format: `{"cue":"brick","responses":["doorstop","paperweight","makeshift hammer",...]}`. The original cue travels with the answers so evaluation can judge appropriateness without external lookup.

In [3]:
m = mod.load("glove-840b-300d")
response = {"cue":"brick",
            "responses":["house","paper weight","build a wall","door stop"]}

v_cue = m.embed_exact(response['cue'])

def score_use(use, v_cue, model):
    tokens = pre.remove_stopwords(pre.strip_marks(use).lower())
    vecs = []
    for w in tokens:
        v = model.embed_exact(w)
        if v is not None:
            vecs.append(v)
    if not vecs:
        return tokens, None
    use_vec = vecs[0]
    for v in vecs[1:]:
        use_vec = use_vec * v
    sim = float(np.dot(v_cue, use_vec) / (
        np.linalg.norm(v_cue) * np.linalg.norm(use_vec)))
    return tokens, 1.0 - sim

for use in response['responses']:
    tokens, dist = score_use(use, v_cue, m)
    print(use, "->", tokens, "->", dist)
    
dists = []
for use in response['responses']:
    tokens, dist = score_use(use, v_cue, m)
    if dist is not None:
        dists.append(dist)

score = float(np.mean(dists))
print("n_valid:", len(dists))
print("score:", score)

house -> ['house'] -> 0.4590120315551758
paper weight -> ['paper', 'weight'] -> 0.7363103926181793
build a wall -> ['build', 'wall'] -> 0.7225824892520905
door stop -> ['door', 'stop'] -> 0.8534654527902603
n_valid: 4
score: 0.6928425915539265


In [4]:
from evaluate import evaluate

response = {"cue":"brick",
            "responses":["house","paper weight","build a wall","door stop"]}

result = evaluate(response)
print(result["n_valid"])
print(result["score"])
print(result["models"])

4
0.9446802694583312
{'cbow-subs-300d': 0.8946024561300874, 'cbow-ukwac-subs-300d': 0.9157350789755583, 'cbow-baroni-400d': 0.9226424880325794, 'tasa-lsa-300d': 0.9843338067876175, 'glove-6b-300d': 1.0060875173658133}
